# Autonomous Driving — Train KITTI Detector on Google Colab

**What this notebook does:**
1. Downloads the KITTI Object Detection dataset (~12 GB images + 5 MB labels)
2. Explores the dataset (class distribution, sample images)
3. Trains a MobileNetV2 + detection head model (two-stage transfer learning)
4. Evaluates with accuracy, IoU, and visual predictions
5. Saves the model to Google Drive (or downloads it)
6. Lets you upload your own images/videos for inference

**Before you start:**
- Go to **Runtime → Change runtime type → T4 GPU**
- Estimated time: ~30 min total (15 min download + 15 min training)
- Colab provides ~100 GB disk space — plenty for KITTI

**Architecture:** MobileNetV2 (backbone) → Classification Head (Car/Pedestrian/Cyclist) + BBox Regression Head

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 1: Install dependencies & check GPU
# ════════════════════════════════════════════════════════════
import tensorflow as tf
import sys

print(f"Python:     {sys.version.split()[0]}")
print(f"TensorFlow: {tf.__version__}")

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for g in gpus:
        print(f"GPU:        {g.name}")
    print("\nGPU is ready! Training will be fast.")
else:
    print("\nWARNING: No GPU detected!")
    print("Go to Runtime -> Change runtime type -> T4 GPU")
    print("Training on CPU will be very slow (~8 hours vs 15 min).")

## Step 1: Download the KITTI Dataset

KITTI Object Detection provides:
- **7,481 training images** with bounding box labels (Cars, Pedestrians, Cyclists)
- **7,518 testing images** without labels (for official benchmark submission)

We download from the official S3 mirror. The images zip is ~12 GB — this takes about 5-10 minutes on Colab's connection.

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 2: Download and extract KITTI dataset
# ════════════════════════════════════════════════════════════
import os

DATA_DIR = '/content/kitti'
os.makedirs(DATA_DIR, exist_ok=True)

IMAGES_URL = 'https://s3.eu-central-1.amazonaws.com/avg-kitti/data_object_image_2.zip'
LABELS_URL = 'https://s3.eu-central-1.amazonaws.com/avg-kitti/data_object_label_2.zip'

IMAGES_ZIP = os.path.join(DATA_DIR, 'data_object_image_2.zip')
LABELS_ZIP = os.path.join(DATA_DIR, 'data_object_label_2.zip')

# Download labels (small, fast)
if not os.path.exists(LABELS_ZIP):
    print('Downloading labels (~5 MB)...')
    !wget -q --show-progress -O {LABELS_ZIP} {LABELS_URL}
else:
    print('Labels zip already downloaded.')

# Download images (large, ~12 GB)
if not os.path.exists(IMAGES_ZIP):
    print('\nDownloading images (~12 GB) — this takes 5-10 minutes...')
    !wget -q --show-progress -O {IMAGES_ZIP} {IMAGES_URL}
else:
    print('Images zip already downloaded.')

print('\nDownload complete!')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 3: Extract the dataset
# ════════════════════════════════════════════════════════════
import zipfile

TRAIN_IMG_DIR = os.path.join(DATA_DIR, 'training', 'image_2')
TRAIN_LBL_DIR = os.path.join(DATA_DIR, 'training', 'label_2')

# Extract labels
if not os.path.isdir(TRAIN_LBL_DIR) or len(os.listdir(TRAIN_LBL_DIR)) < 7000:
    print('Extracting labels...')
    with zipfile.ZipFile(LABELS_ZIP, 'r') as z:
        z.extractall(DATA_DIR)
    print(f'  Labels: {len(os.listdir(TRAIN_LBL_DIR)):,} files')
else:
    print(f'Labels already extracted: {len(os.listdir(TRAIN_LBL_DIR)):,} files')

# Extract images
if not os.path.isdir(TRAIN_IMG_DIR) or len(os.listdir(TRAIN_IMG_DIR)) < 7000:
    print('Extracting images (this takes a few minutes)...')
    with zipfile.ZipFile(IMAGES_ZIP, 'r') as z:
        z.extractall(DATA_DIR)
    print(f'  Training images: {len(os.listdir(TRAIN_IMG_DIR)):,} files')
else:
    print(f'Images already extracted: {len(os.listdir(TRAIN_IMG_DIR)):,} files')

# Verify
n_imgs = len([f for f in os.listdir(TRAIN_IMG_DIR) if f.endswith('.png')])
n_lbls = len([f for f in os.listdir(TRAIN_LBL_DIR) if f.endswith('.txt')])
print(f'\nDataset ready: {n_imgs:,} images, {n_lbls:,} labels')

# Clean up zips to save disk space (optional)
# os.remove(IMAGES_ZIP)
# os.remove(LABELS_ZIP)
# print('Zip files removed to save space.')

## Step 2: Explore the Dataset

Before training, let's understand what we're working with:
- What object classes are in the labels?
- How many of each class?
- What do the images look like with ground truth boxes?

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 4: Configuration + KITTI parser
# ════════════════════════════════════════════════════════════
import numpy as np
import cv2
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from collections import Counter

np.random.seed(42)
tf.random.set_seed(42)

# --- Configuration ---
IMG_HEIGHT = 224
IMG_WIDTH  = 224
BATCH_SIZE = 32  # Colab GPUs can handle larger batches

CLASSES = ['Car', 'Pedestrian', 'Cyclist']
CLASS_TO_ID = {name: i for i, name in enumerate(CLASSES)}
ID_TO_CLASS = {i: name for i, name in enumerate(CLASSES)}
NUM_CLASSES = len(CLASSES)

KITTI_CLASS_MAP = {
    'Car': 'Car', 'Van': 'Car',
    'Pedestrian': 'Pedestrian', 'Person_sitting': 'Pedestrian',
    'Cyclist': 'Cyclist',
    'Truck': None, 'Tram': None, 'Misc': None, 'DontCare': None,
}

CLASS_COLORS = {'Car': '#FF4444', 'Pedestrian': '#4488FF', 'Cyclist': '#44DD44'}


def parse_kitti_label(label_path):
    """Parse a KITTI label file into a list of object dicts."""
    objects = []
    with open(label_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 15:
                continue
            mapped = KITTI_CLASS_MAP.get(parts[0])
            if mapped is None:
                continue
            objects.append({
                'class_name': mapped,
                'class_id':   CLASS_TO_ID[mapped],
                'bbox':       [float(parts[4]), float(parts[5]),
                               float(parts[6]), float(parts[7])],
                'truncated':  float(parts[1]),
                'occluded':   int(parts[2]),
            })
    return objects


def calculate_iou(box1, box2):
    """IoU between two [x1, y1, x2, y2] boxes."""
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    intersection = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union = area1 + area2 - intersection
    return intersection / union if union > 0 else 0.0


print('Configuration loaded.')
print(f'Classes: {CLASSES}')
print(f'Input size: {IMG_WIDTH}x{IMG_HEIGHT}')
print(f'Batch size: {BATCH_SIZE}')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 5: Parse all labels and gather statistics
# ════════════════════════════════════════════════════════════
image_files = sorted([f for f in os.listdir(TRAIN_IMG_DIR) if f.endswith('.png')])

all_objects = []
samples = []  # (image_path, class_id, raw_bbox_pixels)
skipped = 0

for img_file in image_files:
    img_id = img_file.replace('.png', '')
    label_path = os.path.join(TRAIN_LBL_DIR, f'{img_id}.txt')

    if not os.path.exists(label_path):
        skipped += 1
        continue

    objects = parse_kitti_label(label_path)
    all_objects.extend(objects)

    if objects:
        # Pick the LARGEST object by bbox area
        largest = max(objects, key=lambda o:
            (o['bbox'][2] - o['bbox'][0]) * (o['bbox'][3] - o['bbox'][1]))
        img_path = os.path.join(TRAIN_IMG_DIR, img_file)
        samples.append((img_path, largest['class_id'], largest['bbox']))
    else:
        skipped += 1

class_counts = Counter(o['class_name'] for o in all_objects)
sample_counts = Counter(s[1] for s in samples)

print(f'Total annotated objects: {len(all_objects):,}')
print(f'Valid training samples:  {len(samples):,}')
print(f'Skipped (no objects):    {skipped}')
print()
print('Object counts by class:')
for cls in CLASSES:
    print(f'  {cls:12s}: {class_counts.get(cls, 0):5,}')
print()
print('Training samples by class (largest object per image):')
for cls in CLASSES:
    print(f'  {cls:12s}: {sample_counts.get(CLASS_TO_ID[cls], 0):5,}')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 6: Visualize sample images with ground truth
# ════════════════════════════════════════════════════════════
fig, axes = plt.subplots(2, 3, figsize=(18, 9))
fig.suptitle('KITTI — Sample Images with Ground Truth Boxes', fontsize=14, fontweight='bold')

sample_indices = [0, 10, 50, 100, 300, 1000]
for ax, idx in zip(axes.flat, sample_indices):
    if idx >= len(image_files):
        ax.axis('off')
        continue
    img_file = image_files[idx]
    img_id = img_file.replace('.png', '')
    img = cv2.imread(os.path.join(TRAIN_IMG_DIR, img_file))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    label_path = os.path.join(TRAIN_LBL_DIR, f'{img_id}.txt')
    objects = parse_kitti_label(label_path) if os.path.exists(label_path) else []

    ax.imshow(img)
    for obj in objects:
        x1, y1, x2, y2 = obj['bbox']
        color = CLASS_COLORS[obj['class_name']]
        rect = Rectangle((x1, y1), x2-x1, y2-y1, linewidth=2,
                         edgecolor=color, facecolor='none')
        ax.add_patch(rect)
        ax.text(x1, y1-3, obj['class_name'], fontsize=7, color='white',
                fontweight='bold', bbox=dict(boxstyle='round,pad=0.2',
                facecolor=color, alpha=0.8))
    ax.set_title(f'Image {img_id}  ({len(objects)} objects)', fontsize=10)
    ax.axis('off')

plt.tight_layout()
plt.show()

# Class distribution bar chart
fig, ax = plt.subplots(figsize=(7, 4))
colors = [CLASS_COLORS[cls] for cls in CLASSES]
counts = [class_counts.get(cls, 0) for cls in CLASSES]
bars = ax.bar(CLASSES, counts, color=colors, edgecolor='black')
for bar, c in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
            f'{c:,}', ha='center', fontweight='bold')
ax.set_title('Object Class Distribution (All Annotations)', fontweight='bold')
ax.set_ylabel('Count')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## Step 3: Build the Detection Model

**Architecture:**
- **Backbone:** MobileNetV2 pre-trained on ImageNet (frozen initially)
- **Classification head:** Dense → Softmax over 3 classes (Car/Pedestrian/Cyclist)
- **BBox regression head:** Dense → Sigmoid for normalized [x1, y1, x2, y2]

**Loss:** CrossEntropy (classification, weight=1) + MSE (bbox, weight=5)

This is a **single-object localizer** — it detects the main object per image.
See MODEL_ARCHITECTURE.md for discussion of multi-object alternatives (YOLO, SSD, etc.).

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 7: Data generator
# ════════════════════════════════════════════════════════════
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau


class KITTIDataGenerator(keras.utils.Sequence):
    """Loads KITTI images on-the-fly. Normalizes bboxes per-image."""

    def __init__(self, samples, batch_size=32, img_size=(IMG_WIDTH, IMG_HEIGHT),
                 shuffle=True, augment=False):
        self.samples    = list(samples)
        self.batch_size = batch_size
        self.img_size   = img_size
        self.shuffle    = shuffle
        self.augment    = augment
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.samples) / self.batch_size))

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.samples)

    def __getitem__(self, idx):
        batch = self.samples[idx * self.batch_size : (idx + 1) * self.batch_size]
        images, classes, bboxes = [], [], []

        for img_path, class_id, raw_bbox in batch:
            img = cv2.imread(img_path)
            if img is None:
                continue
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            h, w = img.shape[:2]

            # Normalize bbox to [0, 1]
            x1, y1, x2, y2 = raw_bbox
            bbox_norm = [x1/w, y1/h, x2/w, y2/h]

            # Resize + scale pixels
            img = cv2.resize(img, self.img_size).astype(np.float32) / 255.0

            # Augmentation
            if self.augment:
                if np.random.random() > 0.5:
                    img = np.fliplr(img).copy()
                    bx1, by1, bx2, by2 = bbox_norm
                    bbox_norm = [1.0-bx2, by1, 1.0-bx1, by2]
                if np.random.random() > 0.5:
                    img = np.clip(img * np.random.uniform(0.7, 1.3), 0, 1)

            images.append(img)
            classes.append(class_id)
            bboxes.append(bbox_norm)

        return (
            np.array(images, dtype=np.float32),
            {
                'classification': np.array(classes, dtype=np.int32),
                'bbox':           np.array(bboxes, dtype=np.float32),
            }
        )


print('Data generator defined.')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 8: Build model + prepare data
# ════════════════════════════════════════════════════════════

# --- Train/Val split ---
np.random.shuffle(samples)
split_idx = int(len(samples) * 0.8)
train_samples = samples[:split_idx]
val_samples   = samples[split_idx:]

print(f'Training:   {len(train_samples):,} samples')
print(f'Validation: {len(val_samples):,} samples')

train_gen = KITTIDataGenerator(train_samples, batch_size=BATCH_SIZE, shuffle=True, augment=True)
val_gen   = KITTIDataGenerator(val_samples, batch_size=BATCH_SIZE, shuffle=False, augment=False)

print(f'Batches per epoch: {len(train_gen)} train, {len(val_gen)} val')
print()

# --- Build model ---
base_model = MobileNetV2(weights='imagenet', include_top=False,
                         input_shape=(IMG_HEIGHT, IMG_WIDTH, 3))
base_model.trainable = False

inputs = keras.Input(shape=(IMG_HEIGHT, IMG_WIDTH, 3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.3)(x)

cls_head  = layers.Dense(NUM_CLASSES, activation='softmax', name='classification')(x)
bbox_head = layers.Dense(4, activation='sigmoid', name='bbox')(x)

model = keras.Model(inputs=inputs,
                    outputs={'classification': cls_head, 'bbox': bbox_head},
                    name='KITTI_Detector')

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss={'classification': 'sparse_categorical_crossentropy', 'bbox': 'mse'},
    loss_weights={'classification': 1.0, 'bbox': 5.0},
    metrics={'classification': ['accuracy'], 'bbox': ['mae']}
)

print(f'Model: {model.count_params():,} parameters')
print(f'Backbone (frozen): {base_model.count_params():,} parameters')
model.summary()

## Step 4: Train the Model

Two-stage training:
1. **Stage 1** — Backbone frozen, lr=0.001: teaches the detection heads
2. **Stage 2** — Top 30 backbone layers unfrozen, lr=0.00001: fine-tunes features for driving scenes

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 9: Stage 1 — Frozen backbone training
# ════════════════════════════════════════════════════════════
print('STAGE 1: Frozen backbone (learning detection heads)')
print(f'  Epochs: up to 15 (EarlyStopping patience=5)')
print(f'  Learning rate: 0.001')
print()

callbacks_s1 = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=1),
]

history1 = model.fit(
    train_gen, validation_data=val_gen,
    epochs=15, callbacks=callbacks_s1, verbose=1
)

stage1_epochs = len(history1.history['loss'])
print(f'\nStage 1 complete: {stage1_epochs} epochs')
print(f'Best val_loss: {min(history1.history["val_loss"]):.4f}')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 10: Stage 2 — Fine-tuning
# ════════════════════════════════════════════════════════════
print('STAGE 2: Fine-tuning top 30 backbone layers')
print(f'  Learning rate: 0.00001 (100x lower)')
print()

FINE_TUNE_LAYERS = 30
base_model.trainable = True
for layer in base_model.layers[:len(base_model.layers) - FINE_TUNE_LAYERS]:
    layer.trainable = False

trainable = sum(1 for l in base_model.layers if l.trainable)
frozen = sum(1 for l in base_model.layers if not l.trainable)
print(f'  Backbone: {trainable} trainable, {frozen} frozen layers')

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss={'classification': 'sparse_categorical_crossentropy', 'bbox': 'mse'},
    loss_weights={'classification': 1.0, 'bbox': 5.0},
    metrics={'classification': ['accuracy'], 'bbox': ['mae']}
)

callbacks_s2 = [
    EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True, verbose=1),
]

history2 = model.fit(
    train_gen, validation_data=val_gen,
    epochs=10, callbacks=callbacks_s2, verbose=1
)

stage2_epochs = len(history2.history['loss'])
print(f'\nStage 2 complete: {stage2_epochs} epochs')
print(f'\nTotal training: {stage1_epochs + stage2_epochs} epochs')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 11: Plot training curves
# ════════════════════════════════════════════════════════════
# Combine both stage histories
full = {}
for key in history1.history:
    full[key] = history1.history[key] + history2.history.get(key, [])

epochs_range = range(1, len(full['loss']) + 1)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Training Curves  (Stage 1: Frozen | Stage 2: Fine-tuning)',
             fontsize=13, fontweight='bold')

# Total loss
axes[0].plot(epochs_range, full['loss'], 'b-', label='Train', linewidth=1.5)
axes[0].plot(epochs_range, full['val_loss'], 'r-', label='Val', linewidth=1.5)
axes[0].axvline(x=stage1_epochs + 0.5, color='gray', ls='--', alpha=0.5, label='Fine-tune')
axes[0].set_title('Total Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Classification accuracy
axes[1].plot(epochs_range, full['classification_accuracy'], 'b-', label='Train')
axes[1].plot(epochs_range, full['val_classification_accuracy'], 'r-', label='Val')
axes[1].axvline(x=stage1_epochs + 0.5, color='gray', ls='--', alpha=0.5)
axes[1].set_title('Classification Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].legend()
axes[1].grid(alpha=0.3)

# Bbox MAE
axes[2].plot(epochs_range, full['bbox_mae'], 'b-', label='Train')
axes[2].plot(epochs_range, full['val_bbox_mae'], 'r-', label='Val')
axes[2].axvline(x=stage1_epochs + 0.5, color='gray', ls='--', alpha=0.5)
axes[2].set_title('BBox Mean Absolute Error')
axes[2].set_xlabel('Epoch')
axes[2].legend()
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Step 5: Evaluate on Validation Set

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 12: Evaluate — accuracy, IoU, per-class stats
# ════════════════════════════════════════════════════════════
correct = 0
total = 0
ious = []
per_cls_correct = Counter()
per_cls_total = Counter()

print('Evaluating on validation set...')
for img_path, gt_class_id, gt_bbox in val_samples:
    img = cv2.imread(img_path)
    if img is None:
        continue
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img_rgb.shape[:2]

    img_input = cv2.resize(img_rgb, (IMG_WIDTH, IMG_HEIGHT)).astype(np.float32) / 255.0
    preds = model.predict(np.expand_dims(img_input, 0), verbose=0)

    pred_cls = int(np.argmax(preds['classification'][0]))
    pb = preds['bbox'][0]
    pred_bbox = [pb[0]*w, pb[1]*h, pb[2]*w, pb[3]*h]

    is_correct = (pred_cls == gt_class_id)
    correct += int(is_correct)
    total += 1
    per_cls_total[gt_class_id] += 1
    if is_correct:
        per_cls_correct[gt_class_id] += 1

    ious.append(calculate_iou(gt_bbox, pred_bbox))

print(f'\nRESULTS ({total} samples)')
print(f'{"-"*50}')
print(f'Classification accuracy: {correct/total:.1%}')
print()
print('Per-class accuracy:')
for cid in range(NUM_CLASSES):
    ct = per_cls_total[cid]
    cc = per_cls_correct[cid]
    if ct > 0:
        print(f'  {ID_TO_CLASS[cid]:12s}: {cc}/{ct}  ({cc/ct:.1%})')
print()
print(f'Bounding box IoU:')
print(f'  Mean IoU:   {np.mean(ious):.3f}')
print(f'  Median IoU: {np.median(ious):.3f}')
print(f'  IoU > 0.50: {sum(1 for i in ious if i > 0.5)}/{total} ({sum(1 for i in ious if i > 0.5)/total:.1%})')
print(f'  IoU > 0.75: {sum(1 for i in ious if i > 0.75)}/{total} ({sum(1 for i in ious if i > 0.75)/total:.1%})')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 13: Visualize predictions vs ground truth
# ════════════════════════════════════════════════════════════
num_vis = min(6, len(val_samples))
vis_idx = np.linspace(0, len(val_samples)-1, num_vis, dtype=int)

fig, axes = plt.subplots(2, 3, figsize=(18, 9))
fig.suptitle('Predictions vs Ground Truth  (Green=GT, Red=Pred)',
             fontsize=13, fontweight='bold')

for ax, idx in zip(axes.flat, vis_idx):
    img_path, gt_cid, gt_bbox = val_samples[idx]
    img = cv2.imread(img_path)
    if img is None:
        ax.axis('off'); continue
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img_rgb.shape[:2]

    img_in = cv2.resize(img_rgb, (IMG_WIDTH, IMG_HEIGHT)).astype(np.float32) / 255.0
    preds = model.predict(np.expand_dims(img_in, 0), verbose=0)
    pred_cls = int(np.argmax(preds['classification'][0]))
    pred_conf = float(preds['classification'][0][pred_cls])
    pb = preds['bbox'][0]
    pred_bbox = [pb[0]*w, pb[1]*h, pb[2]*w, pb[3]*h]

    iou = calculate_iou(gt_bbox, pred_bbox)

    ax.imshow(img_rgb)

    # GT box (green dashed)
    gx1, gy1, gx2, gy2 = gt_bbox
    ax.add_patch(Rectangle((gx1, gy1), gx2-gx1, gy2-gy1,
                           lw=3, edgecolor='#00CC00', facecolor='none', ls='--'))
    # Pred box (red solid)
    px1, py1, px2, py2 = pred_bbox
    ax.add_patch(Rectangle((px1, py1), px2-px1, py2-py1,
                           lw=2, edgecolor='#FF0000', facecolor='none'))

    ax.set_title(f'GT: {ID_TO_CLASS[gt_cid]} | Pred: {ID_TO_CLASS[pred_cls]} ({pred_conf:.0%})\n'
                 f'IoU: {iou:.2f}', fontsize=10)
    ax.axis('off')

plt.tight_layout()
plt.show()

## Step 6: Save the Model

Two options:
1. **Google Drive** — mount your Drive and save there (persists after runtime ends)
2. **Direct download** — download the .keras file to your local machine

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 14: Save model to Google Drive
# ════════════════════════════════════════════════════════════
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Create a folder for the project
save_dir = '/content/drive/MyDrive/Autonomous_Driving'
os.makedirs(save_dir, exist_ok=True)

# Save model
model_path = os.path.join(save_dir, 'av_perception_final.keras')
model.save(model_path)
print(f'Model saved to Google Drive: {model_path}')

# Also save locally for immediate use
model.save('/content/av_perception_final.keras')
print('Also saved locally: /content/av_perception_final.keras')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 15: (Alternative) Direct download to your computer
# ════════════════════════════════════════════════════════════
# Uncomment the lines below to download directly:

# from google.colab import files
# files.download('/content/av_perception_final.keras')

## Step 7: Try Your Own Image!

Upload a photo (traffic scene, dashcam screenshot, etc.) and run detection on it.

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 16: Upload and process your own image
# ════════════════════════════════════════════════════════════
from google.colab import files

print('Upload an image (traffic scene, dashcam photo, etc.):')
uploaded = files.upload()

CLASS_COLORS_CV2 = {
    'Car':        (255, 68, 68),
    'Pedestrian': (68, 136, 255),
    'Cyclist':    (68, 221, 68),
}

for filename in uploaded:
    # Read image
    img = cv2.imread(filename)
    if img is None:
        print(f'Could not read {filename}')
        continue
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img_rgb.shape[:2]

    # Predict
    img_input = cv2.resize(img_rgb, (IMG_WIDTH, IMG_HEIGHT)).astype(np.float32) / 255.0
    preds = model.predict(np.expand_dims(img_input, 0), verbose=0)

    pred_cls = int(np.argmax(preds['classification'][0]))
    pred_conf = float(preds['classification'][0][pred_cls])
    pred_name = ID_TO_CLASS[pred_cls]
    pb = preds['bbox'][0]
    x1, y1, x2, y2 = int(pb[0]*w), int(pb[1]*h), int(pb[2]*w), int(pb[3]*h)

    # Draw
    fig, ax = plt.subplots(1, 1, figsize=(14, 8))
    ax.imshow(img_rgb)
    color = CLASS_COLORS.get(pred_name, '#FFFF00')
    ax.add_patch(Rectangle((x1, y1), x2-x1, y2-y1,
                           lw=3, edgecolor=color, facecolor='none'))
    ax.text(x1, y1-5, f'{pred_name}: {pred_conf:.0%}', fontsize=12,
            color='white', fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.3', facecolor=color, alpha=0.9))
    ax.set_title(f'{filename} — Detected: {pred_name} ({pred_conf:.0%})', fontsize=13)
    ax.axis('off')
    plt.tight_layout()
    plt.show()

    print(f'  {pred_name}: {pred_conf:.1%}  bbox=({x1},{y1},{x2},{y2})')
    print()

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 17: Process a video (upload -> detect -> download)
# ════════════════════════════════════════════════════════════
import time

print('Upload a video file (dashcam footage, traffic clip, etc.):')
uploaded = files.upload()

for filename in uploaded:
    cap = cv2.VideoCapture(filename)
    if not cap.isOpened():
        print(f'Could not open: {filename}')
        continue

    fps = int(cap.get(cv2.CAP_PROP_FPS)) or 30
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    output_name = f'detected_{filename}'
    if not output_name.endswith('.mp4'):
        output_name = output_name.rsplit('.', 1)[0] + '.mp4'

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_name, fourcc, fps, (width, height))

    print(f'Processing: {width}x{height} @ {fps}fps, {total} frames')
    frame_count = 0
    start = time.time()

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # Detect
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        h, w = frame_rgb.shape[:2]
        inp = cv2.resize(frame_rgb, (IMG_WIDTH, IMG_HEIGHT)).astype(np.float32) / 255.0
        preds = model.predict(np.expand_dims(inp, 0), verbose=0)

        pred_cls = int(np.argmax(preds['classification'][0]))
        pred_conf = float(preds['classification'][0][pred_cls])
        pred_name = ID_TO_CLASS[pred_cls]
        pb = preds['bbox'][0]
        x1, y1, x2, y2 = int(pb[0]*w), int(pb[1]*h), int(pb[2]*w), int(pb[3]*h)

        # Draw on frame (BGR)
        if pred_conf > 0.3:
            color = CLASS_COLORS_CV2.get(pred_name, (0, 255, 255))
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            label = f'{pred_name}: {pred_conf:.0%}'
            (lw, lh), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
            cv2.rectangle(frame, (x1, max(y1-lh-10, 0)), (x1+lw+4, y1), color, -1)
            cv2.putText(frame, label, (x1+2, y1-4),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 2)

        out.write(frame)
        frame_count += 1
        if frame_count % 50 == 0:
            pct = frame_count / total * 100 if total > 0 else 0
            elapsed = time.time() - start
            fps_actual = frame_count / elapsed
            print(f'  Frame {frame_count}/{total} ({pct:.0f}%)  FPS: {fps_actual:.1f}')

    cap.release()
    out.release()

    total_time = time.time() - start
    print(f'\nDone! {frame_count} frames in {total_time:.1f}s ({frame_count/total_time:.1f} FPS)')
    print(f'Downloading result...')
    files.download(output_name)

---

## What's Next?

**This model is a single-object localizer** — it detects the main Car/Pedestrian/Cyclist per image.

To upgrade to **multi-object detection** (find ALL objects), consider:
- **YOLOv8/YOLO11**: `pip install ultralytics` and fine-tune on KITTI
- **SSD**: Multi-scale feature maps for varying object sizes
- **Faster R-CNN**: Two-stage detector, slower but more accurate
- **DETR**: Transformer-based, end-to-end

See `MODEL_ARCHITECTURE.md` for detailed comparisons and an extension roadmap.